# Perceptual Complexity Score (PCS)

Score de complejidad armónica basado en correlaciones entre features jSymbolic
y complejidad percibida (encuesta con 43 participantes, 3 niveles MSI).

In [1]:
import pandas as pd
import numpy as np
from pcs import PerceptualComplexityScore, compute_pcs_weights, LONG_TO_SHORT

FEATURES_LONG = [
    'Vertical_Minor_Seconds', 'Vertical_Tritones', 'Vertical_Sevenths',
    'Vertical_Dissonance_Ratio', 'Standard_Triads', 'Seventh_Chords',
    'Non-Standard_Chords', 'Complex_Chords',
    'Distance_Between_Two_Most_Common_Vertical_Intervals',
    'Prevalence_Ratio_of_Two_Most_Common_Vertical_Intervals',
    'Variability_of_Number_of_Simultaneous_Pitch_Classes',
]

## 1. Cargar datos de encuesta

In [2]:
corr = pd.read_csv('survey_correlations.csv')
pvals = pd.read_csv('survey_pvalues.csv')
corr.head(3)

,Feature,Alto,Bajo,Medio
0,7C,0.580,0.239,0.510
1,CC,-0.363,0.356,-0.158
2,DTMCVI,0.325,0.209,0.481


## 2. Derivar pesos perceptuales

In [3]:
weights_raw = compute_pcs_weights('survey_correlations.csv', 'survey_pvalues.csv')
pcs_raw = PerceptualComplexityScore(weights_raw)
pcs_raw.summary()

Feature      Weight  Invert
------------------------------
7C            0.443   False
VNSPC         0.378   False
DTMCVI        0.338   False

Total active features: 3
Total weight sum: 1.159


In [4]:
weights_refined = compute_pcs_weights('survey_correlations.csv', 'survey_pvalues.csv',
                                      exclude_mixed_sign=False)
pcs = PerceptualComplexityScore(weights_refined)
pcs.summary()

Feature      Weight  Invert
------------------------------
7C            0.443   False
VNSPC         0.378   False
DTMCVI        0.338   False
VDR           0.201   False

Total active features: 4
Total weight sum: 1.360


## 3. Demo: aplicar PCS a datos nuevos

Requiere un DataFrame con los 11 features jSymbolic como columnas.

In [5]:
print("Ejemplo de uso:")
print('from pcs import PerceptualComplexityScore')
print()
print('# Opcion 1: desde archivos de encuesta')
print('pcs = PerceptualComplexityScore.from_survey()')
print()
print('# Opcion 2: cargar pesos existentes')
print('import json')
print('with open("pcs_weights.json") as f:')
print('    w = json.load(f)')  
print('pcs = PerceptualComplexityScore(w)')
print()
print('# Aplicar a datos')
print('scores = pcs.compute(mis_datos)')
print('print(scores.mean(), scores.std())')

Ejemplo de uso:
from pcs import PerceptualComplexityScore

# Opcion 1: desde archivos de encuesta
pcs = PerceptualComplexityScore.from_survey()

# Opcion 2: cargar pesos existentes
import json
with open("pcs_weights.json") as f:
    w = json.load(f)
pcs = PerceptualComplexityScore(w)

# Aplicar a datos
scores = pcs.compute(mis_datos)
print(scores.mean(), scores.std())


## 4. Exportar pesos para reuso

In [6]:
import json
serializable = {}
for feat, info in pcs.weights.items():
    if info['weight'] > 0:
        serializable[feat] = {'weight': round(float(info['weight']), 4), 'invert': bool(info['invert'])}

with open('pcs_weights.json', 'w') as f:
    json.dump(serializable, f, indent=2)
    
print(json.dumps(serializable, indent=2))

{
  "Seventh_Chords": {
    "weight": 0.443,
    "invert": false
  },
  "Distance_Between_Two_Most_Common_Vertical_Intervals": {
    "weight": 0.3383,
    "invert": false
  },
  "Vertical_Dissonance_Ratio": {
    "weight": 0.2007,
    "invert": false
  },
  "Variability_of_Number_of_Simultaneous_Pitch_Classes": {
    "weight": 0.3777,
    "invert": false
  }
}
